# Нормализация в pytorch

Зачем это надо:

Изначально считалось, что нормализация борется с Internal Covariate Shift (сдвигом внутреннего ковариата) - распределение активаций на каждом слое меняется в процессе обучения.

Сглаживание ландшафта функции потерь. Нормализация делает поверхность потерь более гладкой и липшицевой. Это позволяет использовать большие learning rate.

Борьба с затухающими/взрывающимися градиентами. Нормализация не дает значениям активаций улетать в бесконечность или стремиться к нулю.

In [589]:
import torch
torch.cuda.is_available()
torch.random.manual_seed(42)

Нотация по осям:

N - batch size

C - channels (каналы/признаки)

S - spatial (пространственная размерность данных: последовательность 1d - L, картинка 2d - H, W и тд)

Формула всегда одна, различие только в осях на которых она применяется

$$ \hat{x} = \frac{x-\mu}{\sqrt{\sigma^2}+\epsilon} $$
$$ y = \gamma \hat{x} + \beta $$

$\epsilon$ здесь для защиты от деления на 0. По дефолту 1e-5

$\gamma$ (scale) и $\beta$ (shift) - обучаемые параметры, включаются в торче параметром affine=True (по дефолту True). Обучаемые параметры здесь нужны для того, чтобы при необходимости нейросеть могла откатить нормализацию.

# Batchnorm

https://docs.pytorch.org/docs/2.13/generated/torch.nn.BatchNorm1d.html

Статистики: вычисляются по осям N и S для каждого канала/признака (всегда dim=[0, все spatial]). 

Применение: CNN, MLP

Особенности: требует большого размера батча для расчета осмысленных статистик


In [590]:
m = torch.nn.BatchNorm1d(5, affine=False, momentum=0)
input = torch.randn(3, 5) * 100
output = m(input)
print(input)
print(output)

tensor([[  33.6690,   12.8809,   23.4462,   23.0333, -112.2856],
        [ -18.6328,  220.8201,  -63.7997,   46.1657,   26.7351],
        [  53.4905,   80.9357,  111.0290, -168.9799,  -98.8960]])
tensor([[ 0.3559, -1.0628, -0.0016,  0.5838, -0.8113],
        [-1.3633,  1.3394, -1.2240,  0.8236,  1.4088],
        [ 1.0074, -0.2766,  1.2255, -1.4074, -0.5975]])


In [591]:
mean = input.mean(dim=0)
std = input.std(dim=0, correction=0)
print(mean)
print(std)

tensor([ 22.8422, 104.8789,  23.5585, -33.2603, -61.4822])
tensor([30.4232, 86.5626, 71.3736, 96.4318, 62.6181])


In [592]:
eps = torch.tensor(1e-5)
out = (input - mean) / (std + eps)
out

tensor([[ 0.3559, -1.0628, -0.0016,  0.5838, -0.8113],
        [-1.3633,  1.3394, -1.2240,  0.8236,  1.4088],
        [ 1.0074, -0.2766,  1.2255, -1.4074, -0.5975]])

# Layernorm

https://docs.pytorch.org/docs/2.13/generated/torch.nn.LayerNorm.html

Статистики: вычисляются по C и S для каждого объекта в батче (можно выбрать, сколько осей с конца: dim=-1, dim=[1, 2] но никогда не трогаем dim=0)

Применение: трансформеры, RNN

Особенности: работает при любом размере батча, в CNN работает хуже BatchNorm тк смешивает информацию между каналами

In [593]:
batch, sentence_length, embedding_dim = 2, 3, 5
embedding = torch.randn(batch, sentence_length, embedding_dim) * 100
layer_norm = torch.nn.LayerNorm((sentence_length, embedding_dim), elementwise_affine=False)
output = layer_norm(embedding)
print(embedding)
print(output)

tensor([[[-138.4674,  -87.1236,  -22.3366,  171.7361,   31.8880],
         [ -42.4519,   30.5721,  -77.4593, -155.7572,   99.5636],
         [ -87.9786,  -60.1142, -127.4151,  212.2785,  138.5161]],

        [[ -44.5852,  144.5134,   85.6413,  221.8076,   52.3166],
         [  34.6647,  -19.7331,  -45.2748,  -77.1778,  -17.2190],
         [  52.3788,    5.6622,   42.6296,   57.5005,  -64.1724]]])
tensor([[[-1.1575, -0.7032, -0.1301,  1.5869,  0.3497],
         [-0.3080,  0.3380, -0.6177, -1.3104,  0.9484],
         [-0.7108, -0.4643, -1.0597,  1.9456,  1.2930]],

        [[-0.9355,  1.4818,  0.7292,  2.4699,  0.3032],
         [ 0.0776, -0.6178, -0.9444, -1.3522, -0.5857],
         [ 0.3040, -0.2932,  0.1794,  0.3695, -1.1859]]])


In [594]:
mean = embedding.mean(dim=[1, 2], keepdim=True)
std = embedding.std(dim=[1, 2], keepdim=True, correction=0)
print(mean)
print(std)

tensor([[[-7.6366]],

        [[28.5968]]])
tensor([[[113.0326]],

        [[ 78.2246]]])


In [595]:
out = (embedding - mean) / (std + eps)
print(out)

tensor([[[-1.1575, -0.7032, -0.1301,  1.5869,  0.3497],
         [-0.3080,  0.3380, -0.6177, -1.3104,  0.9484],
         [-0.7108, -0.4643, -1.0597,  1.9456,  1.2930]],

        [[-0.9355,  1.4818,  0.7292,  2.4699,  0.3032],
         [ 0.0776, -0.6178, -0.9444, -1.3522, -0.5857],
         [ 0.3040, -0.2932,  0.1794,  0.3695, -1.1859]]])


# Instancenorm

https://docs.pytorch.org/docs/2.13/generated/torch.nn.InstanceNorm1d.html

Статистики: вычисляются по всем пространственным осям, не трогаем батч и каналы

Применение: в основном изображения (Style Transfer (перенос стиля), GAN)

In [596]:
m = torch.nn.InstanceNorm1d(5, affine=False, momentum=0)
input = torch.randn(2, 3, 5) * 100
output = m(input)
print(input)
print(output)

tensor([[[  33.8323,  169.9170,    1.0868,  -33.8742, -134.0679],
         [ -58.5371,   53.6188,   52.4623, -146.9150,  143.3155],
         [  74.3952,  -48.1584, -104.9466,   60.3899,  -31.6508]],

        [[  58.8634,  -89.0457,   40.9813, -145.7029,  -10.2335],
         [ -59.9153,   47.7056,  -16.9332,   23.3225,  403.5634],
         [ 127.9459,   -1.2685,   24.0836,   13.2535,   76.4241]]])
tensor([[[ 0.2676,  1.6441, -0.0636, -0.4173, -1.4308],
         [-0.6680,  0.4448,  0.4333, -1.5449,  1.3347],
         [ 1.2438, -0.5625, -1.3995,  1.0374, -0.3192]],

        [[ 1.1310, -0.7723,  0.9009, -1.5014,  0.2418],
         [-0.8397, -0.1917, -0.5809, -0.3385,  1.9509],
         [ 1.6720, -1.0334, -0.5026, -0.7293,  0.5933]]])


In [597]:
mean = input.mean(dim=-1, keepdim=True)
std = input.std(dim=-1, keepdim=True, correction=0)
print(mean)
print(std)

tensor([[[  7.3788],
         [  8.7889],
         [ -9.9942]],

        [[-29.0275],
         [ 79.5486],
         [ 48.0877]]])
tensor([[[ 98.8606],
         [100.7879],
         [ 67.8483]],

        [[ 77.7105],
         [166.0842],
         [ 47.7618]]])


In [598]:
out = (input - mean) / (std + eps)
out

tensor([[[ 0.2676,  1.6441, -0.0636, -0.4173, -1.4308],
         [-0.6680,  0.4448,  0.4333, -1.5449,  1.3347],
         [ 1.2438, -0.5625, -1.3995,  1.0374, -0.3192]],

        [[ 1.1310, -0.7723,  0.9009, -1.5014,  0.2418],
         [-0.8397, -0.1917, -0.5809, -0.3385,  1.9509],
         [ 1.6720, -1.0334, -0.5026, -0.7293,  0.5933]]])

# Groupnorm

https://docs.pytorch.org/docs/2.13/generated/torch.nn.GroupNorm.html

Статистики: каналы C делятся на G групп. Считается по Spatial и по каналам внутри одной группы.

Применение: Детекция и сегментация (Detectron2, Mask R-CNN), где из-за высокого разрешения картинок батч делают маленьким.
 
Особенности: по сути компромисс между LayerNorm (где группа = все каналы) и InstanceNorm (где группа = 1 канал).

In [599]:
input = torch.randn(20, 6, 10, 10) * 100
channels, groups = 6, 3 # в торче всегда channels кратно groups иначе ошибка

# Separate 6 channels into 3 groups
m = torch.nn.GroupNorm(groups, channels, affine=False)
# Separate 6 channels into 6 groups (equivalent with InstanceNorm)
# m = nn.GroupNorm(6, 6)
# Put all 6 channels into a single group (equivalent with LayerNorm)
# m = nn.GroupNorm(1, 6)
# Activating the module
output = m(input)

In [600]:
out = []
in_group = channels // groups
for i in range(groups):
    tensor_iter = in_group * i
    temp = input[:, tensor_iter:tensor_iter+in_group, :, :]
        
    mean = temp.mean(dim=[1, 2, 3], keepdim=True)
    std = temp.std(dim=[1, 2, 3], keepdim=True, correction=0)
    temp = (temp - mean) / (std + eps)
    
    out.append(temp)
   
out = torch.cat(out, dim=1)             

In [601]:
torch.allclose(out, output, atol=1e-7)

True

In [602]:
# или так
reshaped = input.view(input.size(0), groups, -1, input.size(2), input.size(3))
print(reshaped.shape)
mean = reshaped.mean(dim=[2, 3, 4], keepdim=True)
std = reshaped.std(dim=[2, 3, 4], keepdim=True, correction=0)
print(mean.shape)
x_norm = (reshaped - mean) / (std + eps)
out_vectorized = x_norm.view_as(input)
out_vectorized.shape

torch.Size([20, 3, 2, 10, 10])
torch.Size([20, 3, 1, 1, 1])


torch.Size([20, 6, 10, 10])

In [603]:
torch.allclose(out, out_vectorized)

True

# Сводка по размерностям

BatchNorm1d - (N, C) или (N, C, L) - dim=0 (и dim=2 если есть L)

LayerNorm - (N, C) или (N, C, L) - dim=1 (и dim=2 если есть L)

InstanceNorm1d - (N, C, L) - dim=2

BatchNorm2d - (N, C, H, W) - dim=0, 2, 3

LayerNorm - (N, C, H, W) - dim=1, 2, 3

InstanceNorm2d - (N, C, H, W) - dim=2, 3

GroupNorm - (N, C, H, W) - dim=2, 3 (и частично dim=1 внутри групп)

BatchNorm3d - (N, C, D, H, W) - dim=0, 2, 3, 4

LayerNorm - (N, C, D, H, W) - dim=1, 2, 3, 4

InstanceNorm3d - (N, C, D, H, W) - dim=2, 3, 4

# Как работает allclose

https://docs.pytorch.org/docs/main/generated/torch.allclose.html

В торче allclose проверяет следующее условие для каждой пары элементов:

$$ \vert input_i - other_i \vert \leq atol + rtol * \vert other_i \vert $$

при дефолтных погрешностях atol = 1e-8 и rtol = 1e-5.